# T12: RAG als Tool

Der RAG-Agent in T01 sucht immer, bevor er antwortet. Hier ist das Retrieval stattdessen ein **Tool**: Das LLM entscheidet, wann und mit welcher Frage es sucht. Allgemeine Fragen beantwortet es aus eigenem Wissen, für Fragen zu PrimePack sucht es.

Dieses Notebook führt den Ablauf wie in T10 von Hand aus. T13 setzt dasselbe Tool in einen `ToolAgent`.

Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

## Setup

Zuerst T01 ausführen: Dort entsteht der Vector Store unter `backend/db/data_vs.db`. Ist der Store leer, baut `load_or_build_vector_store()` ihn aus allen Dokumenten auf; das dauert einige Minuten.

In [ ]:
import json
import sys

from loguru import logger

from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from conversational_toolkit.llms.openai import OpenAILLM
from conversational_toolkit.retriever.vectorstore_retriever import VectorStoreRetriever

from ki_kompass_rag.baseline_rag import RETRIEVER_TOP_K, load_or_build_vector_store
from ki_kompass_rag.tool_agents import RetrieveRelevantChunks

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
retriever = VectorStoreRetriever(embedding_model, vector_store, top_k=RETRIEVER_TOP_K)


def text_message(role: Roles, text: str) -> LLMMessage:
    return LLMMessage(role=role, content=[MessageContent(type="text", text=text)])

## Das Retrieval-Tool

`RetrieveRelevantChunks` sucht die besten Chunks zum Argument `query`, das das LLM wählt. Es gibt sie als Text unter `result` zurück und die rohen Chunks unter `_sources`. Schlüssel, die mit `_` beginnen, sind für die Anwendung bestimmt und werden nicht an das LLM geschickt.

In [ ]:
retriever_tool = RetrieveRelevantChunks(retriever)
print(json.dumps(retriever_tool.json_schema(), indent=2))

result = await retriever_tool.call(
    {"query": "Which pallets in our portfolio have a third-party verified EPD?"}
)
print(result["result"][:2000])

## LLM mit dem Tool

Der System-Prompt legt fest, wann das Tool zu verwenden ist und wie mit seinen Ergebnissen umzugehen ist.

`SYSTEM` übersetzt:

> Du bist ein Assistent der PrimePack AG.
> Rufe retrieve_relevant_chunks bei Fragen zu Produkten, Lieferanten oder Dokumenten von PrimePack auf. Beantworte allgemeine Fragen aus deinem eigenen Wissen.
> Wenn du gefundene Chunks verwendest:
> - Verwende sie als einzige Wahrheitsquelle.
> - Wenn sie die Antwort nicht enthalten, sage, dass du es nicht weisst.
> - Nenne für jede Aussage die Quelldatei.
> - Halte die Antwort knapp.

In [ ]:
SYSTEM = """You are an assistant for PrimePack AG.
Call retrieve_relevant_chunks for questions about PrimePack products, suppliers or documents. Answer general questions from your own knowledge.
When you use retrieved chunks:
- Use them as your only source of truth.
- If they do not contain the answer, say that you do not know.
- Cite the source file of every claim.
- Keep the answer concise."""

system_message = text_message(Roles.SYSTEM, SYSTEM)
llm = OpenAILLM(tools=[retriever_tool], tool_choice="auto")

### Allgemeine Frage

Das LLM sollte direkt antworten, ohne Tool-Call.

In [ ]:
response = await llm.generate(
    [
        system_message,
        text_message(
            Roles.USER,
            "What is Einstein's theory of relativity? Answer in 2-3 sentences.",
        ),
    ]
)
print(response.content[0].text)
print(f"Tool calls: {response.tool_calls}")

### Fachliche Frage

Eine Antwort mit Tool-Call enthält noch keinen Antworttext. Das Argument `query` mit der Frage vergleichen: Das LLM formuliert sie oft um.

In [ ]:
messages = [
    system_message,
    text_message(
        Roles.USER, "Which pallets in our portfolio have a third-party verified EPD?"
    ),
]
response = await llm.generate(messages)

print(f"Content: {response.content[0].text!r}")
for tool_call in response.tool_calls or []:
    print(f"{tool_call.function.name}({tool_call.function.arguments})")

Jeden Tool-Call ausführen und sein `result` wie in T10 als `TOOL`-Nachricht zurückschicken. Der letzte Aufruf enthält den System-Prompt, seine Regeln zu Belegen und Quellenangaben gelten also.

In [ ]:
tools = {tool.name: tool for tool in llm.tools or []}
tool_messages = []
for tool_call in response.tool_calls or []:
    result = await tools[tool_call.function.name].call(
        json.loads(tool_call.function.arguments)
    )
    print(f"{tool_call.id}: {len(result['_sources'])} chunks")
    tool_messages.append(
        LLMMessage(
            role=Roles.TOOL,
            name=tool_call.function.name,
            tool_call_id=tool_call.id,
            content=[MessageContent(type="text", text=result["result"])],
        )
    )

final_response = await llm.generate([*messages, response, *tool_messages])
print(final_response.content[0].text)